In [3]:
import pickle
import sys
import json
from pathlib import Path
sys.path.insert(1, str(Path.cwd().parent))
from tensor_hero.model import Transformer
from tensor_hero.inference import full_song_prediction
import torch
import os

song_dir = Path.cwd() / 'inference' / 'songs'
model_to_load = 'cuck'
name_of_song = 'Libidine 8 - Nora Orlandi (youtube)'
outfolder = Path.cwd() / 'generated_songs' / name_of_song
assert not os.path.isdir(outfolder), 'ERROR: Song already exists, 0r is not a directory \nplease enter new name'
os.mkdir(outfolder)
audio_file = song_dir / 'separated.ogg'

# First, let's load the model's parameters and define a Transformer model with them
# Let's take a look at these parameters, they're stored in a dictionary

model_path = Path.cwd() / 'model' / 'model' /'saved_models' / model_to_load
with open(model_path / 'params.pkl', 'rb') as f:
    params = pickle.load(f)
def convert_paths_to_str(obj):
    if isinstance(obj, dict):
        return {k: convert_paths_to_str(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_paths_to_str(i) for i in obj]
    elif isinstance(obj, Path):
        return str(obj)
    else:
        return obj

params_str = convert_paths_to_str(params)
print(json.dumps(params_str, indent=4))

# We use these parameters to define the skeleton of the model, then load the weights into it
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device = {device}')

# Model hyperparameters are derived from the params dictionary
trg_vocab_size = params['trg_vocab_size']
embedding_size = params['embedding_size']
num_heads = params['num_heads']
num_encoder_layers = params['num_encoder_layers']
num_decoder_layers = params['num_decoder_layers']
dropout = params['dropout']
max_len = params['max_trg_len']
forward_expansion = params['embedding_size']*params['forward_expansion']

model = Transformer(
    embedding_size,
    trg_vocab_size,
    num_heads,
    num_encoder_layers,
    num_decoder_layers,
    forward_expansion,
    dropout,
    max_len,
    device,
).to(device)  # Always send the model to the GPU

# Load the weights into the model
model.load_state_dict(torch.load(model_path / (str(model_to_load)+'.pt'), map_location=device))

print(model)


# We have to define some things for the .chart file so it's actually playable
# just some dummy metadata for now
song_metadata = {'Name' : 'libidine8',
                'Artist' : 'Forrest',       # Forrest is the honorary author of all of our output
                'Charter' : 'tensorhero',
                'Offset' : 0,
                'Resolution' : 192,
                'Genre' : 'electronic',
                'MediaType' : 'cd',
                'MusicStream' : 'song.ogg'}

_, raw_output = full_song_prediction(song_path = audio_file,
                         model=model,
                         device=device,
                         sos_idx=432,
                         max_len=500,
                         song_metadata=song_metadata,
                         outfolder=outfolder,
                         RETURN_RAW_OUTPUT=True)

AssertionError: ERROR: Song already exists, enter new name